🔹 Step 1: Environment Setup

In [1]:
# Install required libraries (run once)
# pip install pandas numpy matplotlib scikit-learn tensorflow

In [2]:
!python --version

Python 3.12.12


🔹 Step 2: Import Libraries

In [3]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Input, Bidirectional
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping

🔹 Step 3: Load & Inspect Dataset

In [4]:
df = pd.read_csv("Job_3_Resource_sentiment.csv")
print(df.columns)

Index(['2401', 'Borderlands', 'Positive',
       'im getting on borderlands and i will murder you all ,'],
      dtype='object')


In [5]:
df.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [6]:
df.columns

Index(['2401', 'Borderlands', 'Positive',
       'im getting on borderlands and i will murder you all ,'],
      dtype='object')

In [7]:
df.rename(columns={'Positive': 'sentiment'}, inplace=True)
df.rename(columns={'im getting on borderlands and i will murder you all ,': 'text'}, inplace=True)

In [8]:
df.columns

Index(['2401', 'Borderlands', 'sentiment', 'text'], dtype='object')

In [9]:
df.sample(5)

,2401,Borderlands,sentiment,text
60401,3550,Facebook,Neutral,Facebook Fired An Employee Who Collected Evide...
38279,5364,Hearthstone,Positive,This might actually get me to play Hearthstone...
74596,9186,Nvidia,Positive,from
25819,832,AssassinsCreed,Neutral,Old screenshots from Assassin's Creed Odyssey ...
35728,8134,Microsoft,Irrelevant,you fly into jason aldean a stage with a full ...


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   2401         74681 non-null  int64 
 1   Borderlands  74681 non-null  object
 2   sentiment    74681 non-null  object
 3   text         73995 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [11]:
df = df[['text', 'sentiment']]

In [12]:
df.sample(5)

,text,sentiment
64640,@EAMaddenNFL question because a 91ovr it suppo...,Negative
19534,I decided to give a chance to Shadowlands @War...,Negative
10272,Good to see that Microsoft will release the Xb...,Positive
74578,"New year coming in soon (1660), means that I I...",Positive
70401,Then This could just be dodgy....,Negative


In [13]:
df

,text,sentiment
0,I am coming to the borders and I will kill you...,Positive
1,im getting on borderlands and i will kill you ...,Positive
2,im coming on borderlands and i will murder you...,Positive
3,im getting on borderlands 2 and i will murder ...,Positive
4,im getting into borderlands and i can murder y...,Positive
...,...,...
74676,Just realized that the Windows partition of my...,Positive
74677,Just realized that my Mac window partition is ...,Positive
74678,Just realized the windows partition of my Mac ...,Positive
74679,Just realized between the windows partition of...,Positive


In [14]:
df.shape

(74681, 2)

In [15]:
df.isnull().sum()

,0
text,686
sentiment,0


In [16]:
print(df['sentiment'].value_counts())

sentiment
Negative      22542
Positive      20831
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64


In [17]:
df.duplicated().sum()

np.int64(4909)

🔹 Step 4: Data Cleaning

In [18]:
df.dropna(inplace=True)

In [19]:
df.isnull().sum()

,0
text,0
sentiment,0


In [20]:
df['text'] = df['text'].astype(str)

In [21]:
df['text']

,text
0,I am coming to the borders and I will kill you...
1,im getting on borderlands and i will kill you ...
2,im coming on borderlands and i will murder you...
3,im getting on borderlands 2 and i will murder ...
4,im getting into borderlands and i can murder y...
...,...
74676,Just realized that the Windows partition of my...
74677,Just realized that my Mac window partition is ...
74678,Just realized the windows partition of my Mac ...
74679,Just realized between the windows partition of...


In [22]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    return text.strip()

df['text'] = df['text'].apply(clean_text)

In [23]:
df['text']

,text
0,i am coming to the borders and i will kill you...
1,im getting on borderlands and i will kill you all
2,im coming on borderlands and i will murder you...
3,im getting on borderlands and i will murder y...
4,im getting into borderlands and i can murder y...
...,...
74676,just realized that the windows partition of my...
74677,just realized that my mac window partition is ...
74678,just realized the windows partition of my mac ...
74679,just realized between the windows partition of...


🔹 Step 5: Encode Target Labels

In [24]:
encoder = LabelEncoder()
df['sentiment_encoded'] = encoder.fit_transform(df['sentiment'])

print("Label mapping:")
for i, c in enumerate(encoder.classes_):
    print(i, "->", c)

Label mapping:
0 -> Irrelevant
1 -> Negative
2 -> Neutral
3 -> Positive


In [25]:
encoder.classes_

array(['Irrelevant', 'Negative', 'Neutral', 'Positive'], dtype=object)

In [26]:
df.sample(10)

,text,sentiment,sentiment_encoded
21421,very poor quality of the childs overwatch and ...,Neutral,2
45131,ive got,Neutral,2
29438,i had someone kill these guys they made my tea...,Neutral,2
17823,that s only dead how it be gotta wait till ha...,Negative,1
27171,watching a great assassins creed as valhalla d...,Positive,3
65994,johnson johnson is suspending sales of baby p...,Neutral,2
764,borderlands could we please get a big hot fix ...,Negative,1
43480,bad time to ban pubg only if they quit studyin...,Irrelevant,0
73085,nvidia are keeping the site at cambridge and e...,Negative,1
25938,via game kik assassins creed valhalla sor...,Neutral,2


🔹 Step 6: Text Tokenization & Padding

In [27]:
max_words = 20000
max_length = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(df['text'])

X = pad_sequences(
    tokenizer.texts_to_sequences(df['text']),
    maxlen=max_length,
    padding='post'
)

y = df['sentiment_encoded'].values

In [28]:
tokenizer

In [29]:
df

,text,sentiment,sentiment_encoded
0,i am coming to the borders and i will kill you...,Positive,3
1,im getting on borderlands and i will kill you all,Positive,3
2,im coming on borderlands and i will murder you...,Positive,3
3,im getting on borderlands and i will murder y...,Positive,3
4,im getting into borderlands and i can murder y...,Positive,3
...,...,...,...
74676,just realized that the windows partition of my...,Positive,3
74677,just realized that my mac window partition is ...,Positive,3
74678,just realized the windows partition of my mac ...,Positive,3
74679,just realized between the windows partition of...,Positive,3


In [30]:
X.shape

(73995, 100)

In [31]:
X.dtype

dtype('int32')

In [32]:
y.shape

(73995,)

In [33]:
y.dtype

dtype('int64')

In [34]:
X.ndim

2

In [35]:
y.ndim

1

In [36]:
X.nbytes

29598000

In [37]:
y.nbytes

591960

In [38]:
X

array([[   3,  101,  377, ...,    0,    0,    0],
       [  31,  158,   14, ...,    0,    0,    0],
       [  31,  377,   14, ...,    0,    0,    0],
       ...,
       [  22, 1837,    2, ...,    0,    0,    0],
       [  22, 1837,  693, ...,    0,    0,    0],
       [  22,   32,    2, ...,    0,    0,    0]], dtype=int32)

In [39]:
y

array([3, 3, 3, ..., 3, 3, 3])

🔹 Step 7: Train / Validation / Test Split (MANDATORY)

In [40]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print("\nTrain:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)



Train: (51796, 100)
Validation: (11099, 100)
Test: (11100, 100)


Handle Bias (Class Weight)

In [41]:
print(np.bincount(y_train))

[ 9012 15650 12676 14458]


In [42]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = dict(enumerate(class_weights))
print("\nClass Weights:", class_weights_dict)


Class Weights: {0: np.float64(1.4368619618286729), 1: np.float64(0.8274121405750798), 2: np.float64(1.0215367623856106), 3: np.float64(0.8956287176649605)}


In [43]:
print(np.bincount(y_train))

[ 9012 15650 12676 14458]


🔹 Step 8: Build Deep Learning Model (LSTM)

In [44]:
model = Sequential([
    Input(shape=(max_length,)),
    Embedding(max_words, 128),
    Bidirectional(LSTM(64)),
    Dropout(0.5),
    Dense(4, activation='softmax')
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,659,332 (10.14 MB)

 Trainable params: 2,659,332 (10.14 MB)

 Non-trainable params: 0 (0.00 B)